# NuAncestor Two-Telescope Scheduling Analysis

This notebook evaluates whether two closely located telescope stations can each receive a continuous 30-minute NuAncestor observation every other night.

The telescope pair is selected from the supplied spectrograph list using great-circle distance. By default, only present visible spectrographs on independent telescope resources are considered. FreeFlyer `InContact.txt` and `IsNight.txt` outputs are then used to identify complete nights, continuous valid contact windows, feasible two-station schedules, and cadence violations.

The conservative default assumes that the two 30-minute observations may not overlap. “Every other night” is interpreted as allowing at most one consecutive complete night without a scheduled observation.

Different orbit simulations can be analysed at the same time.

Version 07/2026 by Pedro de S. C. Leonardo

In [2]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re

from pathlib import Path
from IPython.display import display

In [3]:
# Main folder containing the FreeFlyer simulation folders
BASE = Path("All_Results/SchedulingAnalysis")

# Settings to change !!
SPECTROGRAPH_FILE = Path("ListSpectrograph.xlsx")
COUNTRY = "Chile"
INCLUDED_SECTIONS = ["PRESENT VISIBLE"]

AUTO_SELECT_PAIR = True
SELECTED_SPECTROGRAPHS = ["HARPS (S)", "CORALIE"]
REQUIRE_DISTINCT_TELESCOPES = True

CASES = [
    {"name": "a = 28,750 km, i = 0°",
     "semi_major_axis": 28750,
     "inclination": 0,
     "folder": "Results_a28750_i0"}]

DAYS = 365
MIN_OBSERVATION_H = 0.5  # 30 minutes
MAX_CONSECUTIVE_MISSED_NIGHTS = 1  # Every-other-night requirement

# Conservative setting: the two telescope slots cannot overlap
ALLOW_SIMULTANEOUS_OBSERVATIONS = False
COMMON_NIGHT_MODE = "intersection"  # "intersection" or "union"
EXCLUDE_PARTIAL_EDGE_NIGHTS = True

# Columns after elapsed time in the FreeFlyer output files
STATION_COLUMNS = [1, 2]

SELECTED_CASE_NAME = CASES[0]["name"]
PLOT_START_NIGHT = 1
PLOT_END_NIGHT = 60

SAVE_TABLES = False
SAVE_FIGURES = False

In [4]:
SECTION_LABELS = ["INFRA-RED", "RED", "FUTURE", "NO"]

# Corrections for malformed coordinate strings in the spreadsheet
COORDINATE_OVERRIDES = {
    "CHIRON": (-30.169286, -70.806789),
    "FIDEOS": (-29.252062, -70.736907),
    "PlatoSpec": (-29.252062, -70.736907)}

# Instruments sharing one telescope are not independent stations
RESOURCE_ID_OVERRIDES = {
    "HARPS (S)": "La Silla ESO 3.6 m",
    "NIRPS": "La Silla ESO 3.6 m",
    "FIDEOS": "La Silla 1.52 m",
    "PlatoSpec": "La Silla 1.52 m",
    "ANDES (HIRES)": "ELT",
    "CODEX": "ELT"}


def load_spectrographs(path):
    """Read the spectrograph table and retain its section labels."""
    raw = pd.read_excel(path, sheet_name="Sheet1", header=None)
    first_column = raw.iloc[:, 0].astype(str).str.strip()
    header_index = raw.index[first_column == "Spectrograph"][0]
    columns = raw.iloc[header_index].tolist()

    rows = []
    section = "PRESENT VISIBLE"

    for i in range(header_index + 1, len(raw)):
        first_value = raw.iloc[i, 0]

        if pd.isna(first_value):
            continue

        first_value = str(first_value).strip()

        if first_value in SECTION_LABELS:
            section = first_value

            if section == "NO":
                break

            continue

        values = raw.iloc[i, :len(columns)].tolist()
        row = dict(zip(columns, values))
        row["Section"] = section
        rows.append(row)

    return pd.DataFrame(rows)


def dms_to_decimal(value):
    """Convert the coordinate formats used in the spreadsheet to degrees."""
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float)):
        return float(value)

    text = str(value).strip()

    if "http" in text.lower():
        return np.nan

    sign = -1 if text.lstrip().startswith("-") else 1
    numbers = re.findall(r"\d+(?:\.\d+)?", text)

    if not numbers:
        return np.nan

    degrees = float(numbers[0])
    minutes = float(numbers[1]) if len(numbers) >= 2 else 0
    seconds = float(numbers[2]) if len(numbers) >= 3 else 0

    return sign * (degrees + minutes / 60 + seconds / 3600)


def haversine_distance(lat1, lon1, lat2, lon2):
    """Return great-circle distance between two coordinates [km]."""
    earth_radius = 6371.0
    lat1, lon1, lat2, lon2 = np.radians([lat1, lon1, lat2, lon2])

    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1

    a = (np.sin(delta_lat / 2)**2
         + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2)**2)

    return 2 * earth_radius * np.arcsin(np.sqrt(a))


def telescope_resource(row):
    """Return an identifier for the independently schedulable telescope."""
    name = row["Spectrograph"]

    if name in RESOURCE_ID_OVERRIDES:
        return RESOURCE_ID_OVERRIDES[name]

    return f"{row['Observatory Name']} | {row['Telescope Name']}"


spectrographs = load_spectrographs(SPECTROGRAPH_FILE)

spectrographs["Latitude [deg]"] = spectrographs["Latitudes"].apply(
    dms_to_decimal)
spectrographs["Longitude [deg]"] = spectrographs["Longitudes"].apply(
    dms_to_decimal)

for name, (latitude, longitude) in COORDINATE_OVERRIDES.items():
    mask = spectrographs["Spectrograph"] == name
    spectrographs.loc[mask, "Latitude [deg]"] = latitude
    spectrographs.loc[mask, "Longitude [deg]"] = longitude

spectrographs["Resource ID"] = spectrographs.apply(
    telescope_resource, axis=1)

country_spectrographs = spectrographs[
    spectrographs["Observatory Name"].str.contains(
        COUNTRY, case=False, na=False)
    & spectrographs["Section"].isin(INCLUDED_SECTIONS)
    & spectrographs["Latitude [deg]"].notna()
    & spectrographs["Longitude [deg]"].notna()].copy()

pair_rows = []

for i in range(len(country_spectrographs)):
    for j in range(i + 1, len(country_spectrographs)):
        first = country_spectrographs.iloc[i]
        second = country_spectrographs.iloc[j]

        if (REQUIRE_DISTINCT_TELESCOPES
            and first["Resource ID"] == second["Resource ID"]):
            continue

        pair_rows.append({
            "Station 1": first["Spectrograph"],
            "Station 2": second["Spectrograph"],
            "Distance [km]": haversine_distance(
                first["Latitude [deg]"], first["Longitude [deg]"],
                second["Latitude [deg]"], second["Longitude [deg]"])})

pair_distances = (
    pd.DataFrame(pair_rows)
    .sort_values("Distance [km]")
    .reset_index(drop=True))

if pair_distances.empty:
    raise ValueError("No valid telescope pair was found with the selected filters.")

if AUTO_SELECT_PAIR:
    selected_names = pair_distances.loc[0, ["Station 1", "Station 2"]].tolist()
else:
    selected_names = SELECTED_SPECTROGRAPHS

selected_stations = (
    country_spectrographs[
        country_spectrographs["Spectrograph"].isin(selected_names)]
    .set_index("Spectrograph")
    .loc[selected_names]
    .reset_index())

selected_pair_distance = haversine_distance(
    selected_stations.loc[0, "Latitude [deg]"],
    selected_stations.loc[0, "Longitude [deg]"],
    selected_stations.loc[1, "Latitude [deg]"],
    selected_stations.loc[1, "Longitude [deg]"])

print("Closest candidate pairs:")
display(pair_distances.head(10).round({"Distance [km]": 3}))

print("\nSelected stations:")
display(selected_stations[[
    "Spectrograph", "Observatory Name", "Telescope Name",
    "Latitude [deg]", "Longitude [deg]", "Resource ID"]].style.hide(axis="index"))

FileNotFoundError: [Errno 2] No such file or directory: 'ListSpectrograph.xlsx'

### Required FreeFlyer simulation

Create two independent GroundStation objects using the selected coordinates above and the correct observatory altitudes. The FreeFlyer run should use the same orbit, epoch, propagation step, elevation limit, Sun-elevation limit, and tracking-speed limits for both stations.

The analysis expects:

- `InContact.txt`: elapsed time followed by the valid-contact Boolean for Station 1 and Station 2.
- `IsNight.txt`: elapsed time followed by the nighttime Boolean for Station 1 and Station 2.
- `InContact` should already include geometric visibility, nighttime, elevation/airmass, elevation-speed, and azimuth-speed constraints.
- A fixed 30-second step is adequate for an initial analysis. A smaller step can be used to reduce contact-boundary uncertainty.
- A one-year simulation is recommended to include seasonal variation.

The station columns must appear in the same order in both files. The selected column indices can be changed in the settings cell.

In [ ]:
station_names = selected_stations["Spectrograph"].tolist()

freeflyer_setup = pd.DataFrame({
    "FreeFlyer column": ["Station 1", "Station 2"],
    "Spectrograph": station_names,
    "Latitude [deg]": selected_stations["Latitude [deg]"],
    "Longitude [deg]": selected_stations["Longitude [deg]"],
    "Altitude": ["Enter authoritative site altitude",
                 "Enter authoritative site altitude"]})

display(freeflyer_setup.style.hide(axis="index"))

print(
    f"Selected pair distance: {selected_pair_distance:.3f} km\n"
    f"InContact.txt columns: ElapsedTime, {station_names[0]}, {station_names[1]}\n"
    f"IsNight.txt columns: ElapsedTime, {station_names[0]}, {station_names[1]}")

In [ ]:
def read_result(folder, filename):
    """Read one FreeFlyer results file."""
    path = BASE / folder / filename
    return pd.read_csv(path, skiprows=3, sep=r"\s+").to_numpy(dtype=float)


def load_case_results(case):
    """Read and validate the contact and nighttime results for one case."""
    folder = case["folder"]
    contact_data = read_result(folder, "InContact.txt")
    night_data = read_result(folder, "IsNight.txt")

    contact_data = contact_data[
        (contact_data[:, 0] >= 0) & (contact_data[:, 0] < DAYS)]
    night_data = night_data[
        (night_data[:, 0] >= 0) & (night_data[:, 0] < DAYS)]

    if contact_data.shape[1] <= max(STATION_COLUMNS):
        raise ValueError(
            f"InContact.txt in {folder} does not contain the selected columns.")

    if night_data.shape[1] <= max(STATION_COLUMNS):
        raise ValueError(
            f"IsNight.txt in {folder} does not contain the selected columns.")

    contact_time = contact_data[:, 0]
    night_time = night_data[:, 0]

    if len(contact_time) != len(night_time):
        raise ValueError(
            f"InContact.txt and IsNight.txt have different lengths in {folder}.")

    if not np.allclose(contact_time, night_time, rtol=0, atol=1e-10):
        raise ValueError(
            f"InContact.txt and IsNight.txt use different time samples in {folder}.")

    visible = np.column_stack([
        contact_data[:, column] >= 0.5
        for column in STATION_COLUMNS])

    is_night = np.column_stack([
        night_data[:, column] >= 0.5
        for column in STATION_COLUMNS])

    return contact_time, visible, is_night

In [ ]:
def infer_simulation_end(time):
    """Infer the simulation end time from the output samples."""
    if len(time) < 2:
        raise ValueError("At least two output time samples are required.")

    time_steps = np.diff(time)

    if not np.all(time_steps > 0):
        raise ValueError("Output times must be strictly increasing.")

    dt = np.median(time_steps)

    if not np.allclose(time_steps, dt, rtol=0, atol=1e-10):
        raise ValueError("The scheduling analysis requires a fixed output step.")

    return min(DAYS, time[-1] + dt), dt


def get_boolean_index_intervals(state):
    """Return start and end indices for continuous True intervals."""
    state = np.asarray(state, dtype=bool)
    padded = np.r_[False, state, False]

    starts = np.where(np.diff(padded.astype(int)) == 1)[0]
    ends = np.where(np.diff(padded.astype(int)) == -1)[0]

    return starts, ends


def get_complete_nights(common_night):
    """Return complete common-night intervals as index pairs."""
    starts, ends = get_boolean_index_intervals(common_night)
    keep = np.ones(len(starts), dtype=bool)

    if EXCLUDE_PARTIAL_EDGE_NIGHTS and len(starts) > 0:
        if common_night[0]:
            keep[0] = False
        if common_night[-1]:
            keep[-1] = False

    return starts[keep], ends[keep]


def get_candidate_slots(visible, start, end, required_steps):
    """Return all continuous slots satisfying the observation duration."""
    local_starts, local_ends = get_boolean_index_intervals(
        visible[start:end])

    slots = []

    for local_start, local_end in zip(local_starts, local_ends):
        run_start = start + local_start
        run_end = start + local_end

        for slot_start in range(
            run_start, run_end - required_steps + 1):
            slots.append((slot_start, slot_start + required_steps))

    return slots


def choose_joint_slots(first_slots, second_slots):
    """Choose the earliest feasible two-station slot combination."""
    if not first_slots or not second_slots:
        return None

    best = None

    for first_slot in first_slots:
        for second_slot in second_slots:
            if (ALLOW_SIMULTANEOUS_OBSERVATIONS
                or first_slot[1] <= second_slot[0]
                or second_slot[1] <= first_slot[0]):

                key = (
                    max(first_slot[1], second_slot[1]),
                    min(first_slot[0], second_slot[0]),
                    first_slot[0],
                    second_slot[0])

                if best is None or key < best[0]:
                    best = (key, first_slot, second_slot)

    if best is None:
        return None

    return best[1], best[2]


def maximum_continuous_hours(visible, start, end, dt_h):
    """Return the longest continuous valid contact in one night."""
    run_starts, run_ends = get_boolean_index_intervals(
        visible[start:end])

    if len(run_starts) == 0:
        return 0.0

    return np.max(run_ends - run_starts) * dt_h


def build_cadence_plan(feasible_nights):
    """Find the minimum-observation plan satisfying the cadence."""
    states = {0: (0, [])}

    for feasible in feasible_nights:
        next_states = {}

        for missed_nights, (count, plan) in states.items():
            if feasible:
                candidate = (count + 1, plan + [True])
                current = next_states.get(0)

                if current is None or candidate < current:
                    next_states[0] = candidate

            if missed_nights < MAX_CONSECUTIVE_MISSED_NIGHTS:
                new_missed_nights = missed_nights + 1
                candidate = (count, plan + [False])
                current = next_states.get(new_missed_nights)

                if current is None or candidate < current:
                    next_states[new_missed_nights] = candidate

        states = next_states

        if not states:
            return None

    return min(states.values())[1]


def longest_false_run(state):
    """Return the longest consecutive run of False values."""
    false_starts, false_ends = get_boolean_index_intervals(
        ~np.asarray(state, dtype=bool))

    if len(false_starts) == 0:
        return 0

    return int(np.max(false_ends - false_starts))


def elapsed_label(elapsed_days):
    """Format elapsed simulation time as day and clock time."""
    total_minutes = int(round(elapsed_days * 24 * 60))
    day = total_minutes // (24 * 60)
    hour = total_minutes % (24 * 60) // 60
    minute = total_minutes % 60

    return f"Day {day}, {hour:02d}:{minute:02d}"

In [ ]:
def analyse_case(case):
    """Evaluate the two-station scheduling requirement for one orbit."""
    time, visible, is_night = load_case_results(case)
    simulation_end, dt = infer_simulation_end(time)
    dt_h = dt * 24
    required_steps = int(np.ceil(MIN_OBSERVATION_H / dt_h - 1e-12))

    if COMMON_NIGHT_MODE == "intersection":
        common_night = is_night[:, 0] & is_night[:, 1]
    elif COMMON_NIGHT_MODE == "union":
        common_night = is_night[:, 0] | is_night[:, 1]
    else:
        raise ValueError(
            "COMMON_NIGHT_MODE must be 'intersection' or 'union'.")

    night_starts, night_ends = get_complete_nights(common_night)

    if len(night_starts) == 0:
        raise ValueError(f"No complete common nights were found for {case['name']}.")

    rows = []
    chosen_slots = []

    for night_number, (night_start, night_end) in enumerate(
        zip(night_starts, night_ends), start=1):

        first_slots = get_candidate_slots(
            visible[:, 0], night_start, night_end, required_steps)
        second_slots = get_candidate_slots(
            visible[:, 1], night_start, night_end, required_steps)
        joint_slots = choose_joint_slots(first_slots, second_slots)

        chosen_slots.append(joint_slots)

        night_end_time = (
            time[night_end] if night_end < len(time) else simulation_end)

        rows.append({
            "Night": night_number,
            "Night start": elapsed_label(time[night_start]),
            "Night end": elapsed_label(night_end_time),
            f"{station_names[0]} max continuous [h]":
                maximum_continuous_hours(
                    visible[:, 0], night_start, night_end, dt_h),
            f"{station_names[1]} max continuous [h]":
                maximum_continuous_hours(
                    visible[:, 1], night_start, night_end, dt_h),
            f"{station_names[0]} ≥30 min": bool(first_slots),
            f"{station_names[1]} ≥30 min": bool(second_slots),
            "Both individually feasible": bool(first_slots and second_slots),
            "Joint schedule feasible": joint_slots is not None})

    schedule = pd.DataFrame(rows)
    feasible_nights = schedule["Joint schedule feasible"].to_numpy()
    cadence_plan = build_cadence_plan(feasible_nights)
    cadence_satisfied = cadence_plan is not None

    if cadence_plan is None:
        cadence_plan = feasible_nights.tolist()

    schedule["Scheduled"] = cadence_plan
    schedule[f"{station_names[0]} slot"] = ""
    schedule[f"{station_names[1]} slot"] = ""

    for i, scheduled in enumerate(cadence_plan):
        if not scheduled:
            continue

        first_slot, second_slot = chosen_slots[i]

        first_end = (
            time[first_slot[1]] if first_slot[1] < len(time)
            else simulation_end)
        second_end = (
            time[second_slot[1]] if second_slot[1] < len(time)
            else simulation_end)

        schedule.loc[i, f"{station_names[0]} slot"] = (
            f"{elapsed_label(time[first_slot[0]])}–"
            f"{elapsed_label(first_end)}")
        schedule.loc[i, f"{station_names[1]} slot"] = (
            f"{elapsed_label(time[second_slot[0]])}–"
            f"{elapsed_label(second_end)}")

    summary = {
        "Case": case["name"],
        "Selected pair": f"{station_names[0]} + {station_names[1]}",
        "Distance [km]": selected_pair_distance,
        "Analysed complete nights": len(schedule),
        "Both individually feasible [nights]":
            int(schedule["Both individually feasible"].sum()),
        "Joint schedule feasible [nights]":
            int(schedule["Joint schedule feasible"].sum()),
        "Scheduled campaign nights": int(schedule["Scheduled"].sum()),
        "Longest run without joint feasibility [nights]":
            longest_false_run(feasible_nights),
        "Every-other-night requirement satisfied":
            "Yes" if cadence_satisfied else "No"}

    return summary, schedule

In [ ]:
case_summaries = []
case_schedules = {}

for case in CASES:
    summary, schedule = analyse_case(case)
    case_summaries.append(summary)
    case_schedules[case["name"]] = schedule

scheduling_summary = pd.DataFrame(case_summaries)

display(
    scheduling_summary
    .round({"Distance [km]": 3})
    .style.hide(axis="index"))

In [ ]:
selected_schedule = case_schedules[SELECTED_CASE_NAME]

print(f"Scheduled nights for {SELECTED_CASE_NAME}:")
display(
    selected_schedule[selected_schedule["Scheduled"]]
    .reset_index(drop=True)
    .round(3)
    .style.hide(axis="index"))

print("\nNights where a joint schedule is not possible:")
display(
    selected_schedule[~selected_schedule["Joint schedule feasible"]]
    .reset_index(drop=True)
    .round(3)
    .style.hide(axis="index"))

In [ ]:
plot_schedule = selected_schedule[
    (selected_schedule["Night"] >= PLOT_START_NIGHT)
    & (selected_schedule["Night"] <= PLOT_END_NIGHT)]

status_columns = [
    f"{station_names[0]} ≥30 min",
    f"{station_names[1]} ≥30 min",
    "Joint schedule feasible",
    "Scheduled"]

status = plot_schedule[status_columns].to_numpy(dtype=int).T

fig, ax = plt.subplots(figsize=(12, 3))
image = ax.imshow(
    status, aspect="auto", interpolation="nearest",
    extent=[
        plot_schedule["Night"].iloc[0] - 0.5,
        plot_schedule["Night"].iloc[-1] + 0.5,
        len(status_columns) - 0.5, -0.5])

ax.set_title(f"Two-Telescope Scheduling: {SELECTED_CASE_NAME}")
ax.set_xlabel("Night")
ax.set_yticks(range(len(status_columns)))
ax.set_yticklabels(status_columns)
ax.set_xticks(np.arange(
    plot_schedule["Night"].iloc[0],
    plot_schedule["Night"].iloc[-1] + 1,
    max(1, len(plot_schedule) // 10)))
ax.grid(False)

colorbar = fig.colorbar(image, ax=ax, ticks=[0, 1])
colorbar.ax.set_yticklabels(["No", "Yes"])

plt.tight_layout()

if SAVE_FIGURES:
    output = BASE / "scheduling_plots"
    output.mkdir(exist_ok=True)

    plt.savefig(
        output / "two_telescope_schedule.png",
        dpi=300, bbox_inches="tight")

In [ ]:
if SAVE_TABLES:
    output = BASE / "scheduling_tables"
    output.mkdir(exist_ok=True)

    scheduling_summary.to_csv(
        output / "scheduling_summary.csv", index=False)

    for case_name, schedule in case_schedules.items():
        filename = re.sub(r"[^A-Za-z0-9_-]+", "_", case_name).strip("_")
        schedule.to_csv(
            output / f"{filename}_schedule.csv", index=False)